In [ ]:
!pip install segmentation_models_pytorch 

In [ ]:
import segmentation_models_pytorch as smp
print(smp.encoders.get_encoder_names())

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected – check Accelerator setting!')

# Install / upgrade packages (tqdm already present on Kaggle)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'tqdm'])

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
DATASET_NAME = "duality-offroad-segmentation"
BASE_PATH = f"/kaggle/input/{DATASET_NAME}/data"

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR   = os.path.join(BASE_PATH, "val")
TEST_DIR  = os.path.join(BASE_PATH, "test")

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))
print("Test exists:", os.path.exists(TEST_DIR))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
value_map = {
    0: 0,
    100: 1,
    200: 2,
    300: 3,
    500: 4,
    550: 5,
    700: 6,
    800: 7,
    7100: 8,
    10000: 9
}

NUM_CLASSES = len(value_map)
print("Number of classes:", NUM_CLASSES)

In [ ]:
class SegDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_dir = os.path.join(root_dir, "Color_Images")
        self.mask_dir  = os.path.join(root_dir, "Segmentation")
        self.images = sorted(os.listdir(self.image_dir))
        self.transform = transform

    def convert_mask(self, mask):
        mask_new = np.zeros_like(mask)
        for raw_val, new_val in value_map.items():
            mask_new[mask == raw_val] = new_val
        return mask_new

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask = self.convert_mask(mask)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask.long()

In [ ]:
IMAGE_SIZE = 512

train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.6, 1.0)),
    
    A.HorizontalFlip(p=0.3),
    A.VerticalFlip(p=0.2),
    
    A.OneOf([
        A.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.6, hue=0.15),
        A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.5),
    ], p=0.7),

    A.RandomGamma(gamma_limit=(70, 130), p=0.5),

    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
    
    A.MotionBlur(blur_limit=5, p=0.3),

    A.CoarseDropout(max_holes=8, max_height=64, max_width=64, p=0.4),

    A.ToGray(p=0.3),

    A.Normalize(),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
train_ds = SegDataset(TRAIN_DIR, transform=train_transform)
val_ds   = SegDataset(VAL_DIR,   transform=val_transform)
test_ds  = SegDataset(TEST_DIR,  transform=val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    drop_last=True 
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    drop_last=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    drop_last=False
)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="mit_b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
)

model.to(DEVICE)

In [ ]:
from segmentation_models_pytorch import losses as smp_losses

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.softmax(preds, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(targets, NUM_CLASSES)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        intersection = (preds * targets_one_hot).sum(dim=(2,3))
        union = preds.sum(dim=(2,3)) + targets_one_hot.sum(dim=(2,3))

        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

# Use SMP's Lovasz loss – set mode to 'multiclass' for your setup
lovasz_loss = smp_losses.LovaszLoss(mode='multiclass', per_image=False)

def combined_loss(pred, target):
    return 0.4 * ce_loss(pred, target) + \
           0.3 * dice_loss(pred, target) + \
           0.3 * lovasz_loss(pred, target)  # SMP handles softmax internally if needed

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
def compute_iou(pred, target, num_classes):
    pred = torch.argmax(pred, dim=1)
    ious = []

    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            ious.append(np.nan)
        else:
            ious.append((intersection / union).item())

    return np.nanmean(ious)

In [ ]:
EPOCHS = 10
best_iou = 0

# Add this before the training loop
train_losses = []
train_ious = []
train_accs = []
val_losses = []
val_ious = []
val_accs = []

# Modify the training loop to compute additional metrics

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_iou = 0
    train_acc = 0
    num_batches = len(train_loader)

    for images, masks in tqdm(train_loader):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = combined_loss(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        iou = compute_iou(outputs, masks, NUM_CLASSES)
        train_iou += iou

        pred = torch.argmax(outputs, dim=1)
        acc = (pred == masks).float().mean().item()
        train_acc += acc

    train_loss /= num_batches
    train_iou /= num_batches
    train_acc /= num_batches

    train_losses.append(train_loss)
    train_ious.append(train_iou)
    train_accs.append(train_acc)

    scheduler.step()

    model.eval()
    val_loss = 0
    val_iou = 0
    val_acc = 0
    num_batches = len(val_loader)

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            outputs = model(images)
            loss = combined_loss(outputs, masks)
            val_loss += loss.item()

            iou = compute_iou(outputs, masks, NUM_CLASSES)
            val_iou += iou

            pred = torch.argmax(outputs, dim=1)
            acc = (pred == masks).float().mean().item()
            val_acc += acc

    val_loss /= num_batches
    val_iou /= num_batches
    val_acc /= num_batches

    val_losses.append(val_loss)
    val_ious.append(val_iou)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | Val Acc: {val_acc:.4f}")

    if val_iou >= best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "/kaggle/working/best_model1.pth")

print("Best Val IoU:", best_iou)

# After training, plot and save figures
epochs = range(1, EPOCHS + 1)

# Loss plot
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, val_losses, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('/kaggle/working/loss_plot.png')
plt.show()

# IoU plot
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_ious, label='Train IoU')
plt.plot(epochs, val_ious, label='Val IoU')
plt.title('IoU over Epochs')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.legend()
plt.savefig('/kaggle/working/iou_plot.png')
plt.show()

# Pixel Accuracy plot
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_accs, label='Train Pixel Accuracy')
plt.plot(epochs, val_accs, label='Val Pixel Accuracy')
plt.title('Pixel Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Pixel Accuracy')
plt.legend()
plt.savefig('/kaggle/working/acc_plot.png')
plt.show()

# Modify the test evaluation to compute more metrics and save to file
model.load_state_dict(torch.load("/kaggle/working/best_model1.pth"))
model.eval()

test_losses = []
test_ious = []
test_accs = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        # Original prediction
        outputs = model(images)

        # TTA: Horizontal flip
        images_flip = torch.flip(images, dims=[3])
        outputs_flip = model(images_flip)
        outputs_flip = torch.flip(outputs_flip, dims=[3])

        # Average logits
        avg_logits = (outputs + outputs_flip) / 2
        probs = torch.softmax(avg_logits, dim=1)

        loss = combined_loss(avg_logits, masks)
        test_losses.append(loss.item())

        iou = compute_iou(avg_logits, masks, NUM_CLASSES)
        test_ious.append(iou)

        pred = torch.argmax(avg_logits, dim=1)
        acc = (pred == masks).float().mean().item()
        test_accs.append(acc)

mean_test_loss = np.mean(test_losses)
mean_test_iou = np.mean(test_ious)
mean_test_acc = np.mean(test_accs)

with open('/kaggle/working/test_metrics.txt', 'w') as f:
    f.write(f"Mean Test Loss: {mean_test_loss:.4f}\n")
    f.write(f"Mean Test IoU: {mean_test_iou:.4f}\n")
    f.write(f"Mean Test Pixel Accuracy: {mean_test_acc:.4f}\n")

print("Test metrics saved to test_metrics.txt")

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model1.pth"))
model.eval()

test_ious = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        # Original prediction
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        # TTA: Horizontal flip
        images_flip = torch.flip(images, dims=[3])
        outputs_flip = model(images_flip)
        probs_flip = torch.softmax(outputs_flip, dim=1)
        probs_flip = torch.flip(probs_flip, dims=[3])

        # Average probabilities
        avg_probs = (probs + probs_flip) / 2

        # Compute IoU using averaged probabilities
        iou = compute_iou(avg_probs, masks, NUM_CLASSES)
        test_ious.append(iou)

print("Test Mean IoU with TTA:", np.mean(test_ious))